# Detector de somnolencia/fatiga al volante
## Notebook 01 — Preparación de datos (MRL Eye Dataset)

Preparación del conjunto de datos para el clasificador de estado ocular (abierto / cerrado): descarga, etiquetado, normalización, filtrado y partición por persona en entrenamiento, validación y prueba, con validación cruzada agrupada.

La partición se realiza **por persona** y no por imagen, de modo que ninguna persona aparece en más de un conjunto (prevención de *data leakage*).

**Salidas:**
- Conjuntos procesados `mrl_train.npz`, `mrl_val.npz`, `mrl_test.npz` en Google Drive, utilizados por el *Notebook 02 — Entrenamiento*.
- Tablas y figuras de resumen del data-set (`resultados_datos.zip`).

## 1. Descarga del MRL Eye Dataset
84 898 imágenes de ojos de 37 personas (~340 MB). Fuente: Fusek (2018), http://mrl.cs.vsb.cz/eyedataset

In [ ]:
import os, zipfile

URL = 'https://mrl.cs.vsb.cz/data/eyedataset/mrlEyes_2018_01.zip'
ZIP = '/content/mrlEyes.zip'

if not os.path.exists(ZIP):
    !wget -q --show-progress -O {ZIP} {URL}

if not os.path.exists('/content/mrl'):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content/mrl')

print('Descarga y descompresión completas')

## 2. Lectura de etiquetas desde el nombre de archivo
Formato: `s0001_00001_0_0_0_0_0_01.png` → persona, id de imagen, género, lentes, **estado del ojo (0 = cerrado, 1 = abierto)**, reflejos, iluminación, sensor.

In [ ]:
import glob
import numpy as np
import pandas as pd

rutas = sorted(glob.glob('/content/mrl/**/*.png', recursive=True))

filas = []
for r in rutas:
    p = os.path.splitext(os.path.basename(r))[0].split('_')
    if len(p) != 8:
        continue
    filas.append({
        'ruta': r,
        'persona': p[0],
        'genero': int(p[2]),
        'lentes': int(p[3]),
        'estado': int(p[4]),
        'reflejos': int(p[5]),
        'iluminacion': int(p[6]),
        'sensor': p[7],
    })

df = pd.DataFrame(filas)
df['clase'] = df['estado'].map({0: 'cerrado', 1: 'abierto'})

print('Imágenes encontradas:', len(df))
print('Personas:', df['persona'].nunique())
print()
print(df['clase'].value_counts())
print()
print('Proporción cerrados/abiertos: %.3f' % ((df.estado == 0).sum() / (df.estado == 1).sum()))

## 3. Normalización y filtrado
- Lectura en escala de grises.
- Descarte de archivos corruptos o ilegibles.
- Redimensionado uniforme a 64 × 64 píxeles.
- Escalado de intensidades a [0, 1] (se guarda en `uint8` para ahorrar memoria y se divide entre 255 al entrenar).

In [ ]:
import cv2
from tqdm.auto import tqdm

TAM = 64
X = np.zeros((len(df), TAM, TAM), dtype=np.uint8)
validas = np.ones(len(df), dtype=bool)
tam_original = []

for i, r in enumerate(tqdm(df['ruta'].values)):
    img = cv2.imread(r, cv2.IMREAD_GRAYSCALE)
    if img is None or img.size == 0:
        validas[i] = False
        tam_original.append((0, 0))
        continue
    tam_original.append(img.shape)
    X[i] = cv2.resize(img, (TAM, TAM), interpolation=cv2.INTER_AREA)

df['alto_orig'] = [t[0] for t in tam_original]
df['ancho_orig'] = [t[1] for t in tam_original]

print('Archivos descartados (corruptos/ilegibles):', int((~validas).sum()))
df = df[validas].reset_index(drop=True)
X = X[validas]

print('Imágenes válidas:', len(df))
print('Tamaño original: mín %dx%d, máx %dx%d píxeles' % (df.alto_orig.min(), df.ancho_orig.min(), df.alto_orig.max(), df.ancho_orig.max()))
muestra = X[:1000].astype('float32') / 255.0
print('Rango tras normalizar: [%.2f, %.2f]' % (muestra.min(), muestra.max()))

In [ ]:
import matplotlib.pyplot as plt

os.makedirs('/content/resultados', exist_ok=True)

rng = np.random.default_rng(0)
idx = rng.choice(len(df), 5, replace=False)
fig, ax = plt.subplots(2, 5, figsize=(12, 5))
for j, i in enumerate(idx):
    orig = cv2.imread(df.ruta[i], cv2.IMREAD_GRAYSCALE)
    ax[0, j].imshow(orig, cmap='gray')
    ax[0, j].set_title('Original %dx%d' % orig.shape, fontsize=9)
    ax[1, j].imshow(X[i], cmap='gray')
    ax[1, j].set_title('Procesada 64x64 (%s)' % df.clase[i], fontsize=9)
    ax[0, j].axis('off'); ax[1, j].axis('off')
plt.suptitle('Pre-procesamiento: imagen original vs. normalizada')
plt.tight_layout()
plt.savefig('/content/resultados/fig_preprocesamiento.png', dpi=150)
plt.show()

## 4. Partición por persona: entrenamiento 70 % · validación 15 % · prueba 15 %
Cada persona se asigna completa a un solo conjunto. Se asignan primero las personas con más imágenes al conjunto que más lejos está de su proporción objetivo.

In [ ]:
objetivo = {'train': 0.70, 'val': 0.15, 'test': 0.15}
conteo = df.groupby('persona').size().sort_values(ascending=False)
total = len(df)

asignadas = {k: 0 for k in objetivo}
split_de = {}
for persona, n in conteo.items():
    destino = max(objetivo, key=lambda k: objetivo[k] * total - asignadas[k])
    split_de[persona] = destino
    asignadas[destino] += n

df['split'] = df['persona'].map(split_de)

# Verificación: ninguna persona en dos conjuntos
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    comunes = set(df[df.split == a].persona) & set(df[df.split == b].persona)
    assert not comunes, 'Persona repetida entre %s y %s: %s' % (a, b, comunes)
print('OK: ninguna persona aparece en más de un conjunto')

for s in ['train', 'val', 'test']:
    print('%-5s -> personas: %s' % (s, sorted(df[df.split == s].persona.unique())))

In [ ]:
nombres = {'train': 'Entrenamiento', 'val': 'Validación', 'test': 'Prueba'}

resumen = df.groupby('split').agg(
    personas=('persona', 'nunique'),
    imagenes=('ruta', 'size'),
    abiertos=('estado', lambda s: int((s == 1).sum())),
    cerrados=('estado', lambda s: int((s == 0).sum())),
    con_lentes=('lentes', 'sum'),
).reindex(['train', 'val', 'test'])

tot = resumen.sum()
resumen.loc['TOTAL'] = tot
resumen['% del total'] = (100 * resumen.imagenes / tot.imagenes).round(1)
resumen['% cerrados'] = (100 * resumen.cerrados / resumen.imagenes).round(1)
resumen['% con lentes'] = (100 * resumen.con_lentes / resumen.imagenes).round(1)
resumen = resumen.drop(columns='con_lentes').rename(index=nombres)

resumen.to_csv('/content/resultados/tabla_particion.csv', encoding='utf-8-sig')
resumen

In [ ]:
graf = resumen.drop(index='TOTAL')[['abiertos', 'cerrados']]
ax = graf.plot(kind='bar', figsize=(8, 5), color=['#4C9BE8', '#E8744C'], rot=0)
for c in ax.containers:
    ax.bar_label(c, fmt='%d', fontsize=9)
ax.set_ylabel('Número de imágenes')
ax.set_xlabel('')
ax.set_title('Distribución de clases por conjunto (MRL Eye Dataset)')
ax.legend(['Ojo abierto', 'Ojo cerrado'])
plt.tight_layout()
plt.savefig('/content/resultados/fig_distribucion_clases.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(3, 8, figsize=(13, 5.5))
rng = np.random.default_rng(1)
for f, s in enumerate(['train', 'val', 'test']):
    ab = rng.choice(np.where((df.split == s) & (df.estado == 1))[0], 4, replace=False)
    ce = rng.choice(np.where((df.split == s) & (df.estado == 0))[0], 4, replace=False)
    for c, i in enumerate(np.concatenate([ab, ce])):
        ax[f, c].imshow(X[i], cmap='gray')
        ax[f, c].set_title('%s | %s' % (df.persona[i], df.clase[i]), fontsize=8)
        ax[f, c].axis('off')
    ax[f, 0].text(-40, 32, nombres[s], rotation=90, va='center', fontsize=11, weight='bold')
plt.suptitle('Ejemplos por conjunto (4 abiertos, 4 cerrados)')
plt.tight_layout()
plt.savefig('/content/resultados/fig_ejemplos_por_conjunto.png', dpi=150)
plt.show()

## 5. Validación cruzada (Cross-Validation) agrupada por persona
`GroupKFold` con k = 5 sobre entrenamiento + validación. En cada pliegue se valida con personas que el modelo no vio. El conjunto de **prueba queda aparte** y solo se usa una vez al final.

In [ ]:
from sklearn.model_selection import GroupKFold

desarrollo = df[df.split.isin(['train', 'val'])]
gkf = GroupKFold(n_splits=5)
df['fold'] = -1

filas = []
for k, (tr, va) in enumerate(gkf.split(desarrollo, desarrollo.estado, groups=desarrollo.persona), 1):
    parte_tr, parte_va = desarrollo.iloc[tr], desarrollo.iloc[va]
    assert not set(parte_tr.persona) & set(parte_va.persona)
    df.loc[parte_va.index, 'fold'] = k
    filas.append({
        'Pliegue': k,
        'Personas (entrenamiento)': parte_tr.persona.nunique(),
        'Imágenes (entrenamiento)': len(parte_tr),
        'Personas (validación)': parte_va.persona.nunique(),
        'Imágenes (validación)': len(parte_va),
        '% cerrados (validación)': round(100 * (parte_va.estado == 0).mean(), 1),
        'Personas de validación': ', '.join(sorted(parte_va.persona.unique())),
    })

cv = pd.DataFrame(filas).set_index('Pliegue')
cv.to_csv('/content/resultados/tabla_validacion_cruzada.csv', encoding='utf-8-sig')
print('OK: en ningún pliegue se repiten personas entre entrenamiento y validación')
cv

## 6. Exportación
- Tablas y figuras de resumen: `resultados_datos.zip`.
- Conjuntos procesados (`.npz`) y archivo de partición: Google Drive, carpeta `ProyectoSomnolencia/datos`, para el Notebook 02.

In [ ]:
GUARDAR_EN_DRIVE = True

cols = ['persona', 'genero', 'lentes', 'estado', 'clase', 'reflejos', 'iluminacion', 'sensor', 'split', 'fold']
particion = df[cols].copy()
particion.insert(0, 'archivo', df.ruta.map(os.path.basename))
particion.to_csv('/content/resultados/particion_mrl.csv', index=False, encoding='utf-8-sig')

!cd /content && zip -qr resultados_datos.zip resultados
from google.colab import files
files.download('/content/resultados_datos.zip')

if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    destino = '/content/drive/MyDrive/ProyectoSomnolencia/datos'
    os.makedirs(destino, exist_ok=True)
    for s in ['train', 'val', 'test']:
        m = (df.split == s).to_numpy()
        np.savez_compressed('%s/mrl_%s.npz' % (destino, s), X=X[m], y=df.estado.to_numpy()[m], persona=df.persona.to_numpy()[m], fold=df.fold.to_numpy()[m])
    particion.to_csv(destino + '/particion_mrl.csv', index=False, encoding='utf-8-sig')
    print('Datos procesados guardados en', destino)